# 02 — Data Cleaning

**Obiettivo:** Trasformare i dati grezzi in un formato pulito, coerente
e pronto per le analisi.

**Decisioni metodologiche principali:**
- I nomi aziende vengono estratti dai titoli HTML via regex
- I valori monetari (`$100MM`, `$1.2B`) vengono convertiti in float USD
- Le date sono normalizzate in `datetime`
- `Founded = 1900` è un placeholder di scraping → trattato come NaN
- `Sector/Subsector = 'Missing'` → NaN
- Le righe rounds senza dati reali (`Funding Date = NaN`) vengono
  mantenute ma flaggate
- `Total Funding` viene ricavato come somma degli `Amount Raised` per azienda

**Output:**
- `companies_clean.parquet`
- `rounds_clean.parquet`

In [2]:
import pandas as pd
import numpy as np
import re
import warnings
from pathlib import Path
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)

# ── percorsi ──────────────────────────────────────────────────────────────────
DATA_DIR = os.path.join("..", "02. Outputs")   # modifica se necessario
COMP_PATH   = os.path.join("..", "02. Outputs", "aziende_principale.csv")
ROUNDS_PATH = os.path.join("..", "02. Outputs", "aziende_funding_rounds.csv")
OUT_DIR = Path("..", "02. Outputs")   # modifica se necessario


# ── caricamento ───────────────────────────────────────────────────────────────
comp   = pd.read_csv(COMP_PATH)
rounds = pd.read_csv(ROUNDS_PATH)


print(f'Companies raw : {comp.shape}')
print(f'Rounds raw    : {rounds.shape}')

Companies raw : (5370, 8)
Rounds raw    : (21538, 17)


## 1 · Funzioni di utilità

In [3]:
# ── Parsing valori monetari ───────────────────────────────────────────────────
# Converte '$100MM' → 100_000_000.0 | '$1.3B' → 1_300_000_000.0
# '--', 'NA', None → np.nan

_NULL_VALUES = {'NA', '--', '—', '', 'nan', 'null', 'N/A'}
_MONEY_RE    = re.compile(r'^\$?([\d.]+)(B|MM|M|K)?$', re.IGNORECASE)

def parse_money(s):
    if pd.isna(s) or str(s).strip() in _NULL_VALUES:
        return np.nan
    m = _MONEY_RE.match(str(s).strip().replace(',', '').replace(' ', ''))
    if not m:
        return np.nan
    val, suffix = float(m.group(1)), (m.group(2) or '').upper()
    multiplier = {'B': 1e9, 'MM': 1e6, 'M': 1e6, 'K': 1e3}.get(suffix, 1.0)
    return val * multiplier

# Test rapido
tests = [('$60MM', 6e7), ('$1.34B', 1.34e9), ('$500K', 5e5),
         ('$0.12', 0.12), ('--', np.nan), (None, np.nan)]
for inp, expected in tests:
    result = parse_money(inp)
    ok = (np.isnan(result) and np.isnan(expected)) or result == expected
    print(f'  {str(inp):12} → {result}  {"✅" if ok else "❌"}')

  $60MM        → 60000000.0  ✅
  $1.34B       → 1340000000.0  ✅
  $500K        → 500000.0  ✅
  $0.12        → 0.12  ✅
  --           → nan  ✅
  None         → nan  ✅


In [4]:
# ── Cleaning nome azienda ─────────────────────────────────────────────────────
# I titoli HTML di Forge seguono due pattern principali.
# Usiamo una lista di regex in cascata; se nessuna matcha, conserviamo l'originale.

_NAME_PATTERNS = [
    re.compile(r'^Access\s+(.+?)\s+Stock and Financial Details\s*[-|]\s*Forge$', re.I),
    re.compile(r'^Invest and Sell\s+(.+?)\s+Stock\s*[-|]\s*Forge$', re.I),
    re.compile(r'^(.+?)\s+Stock Price[^|]*\|?\s*Forge$', re.I),
    re.compile(r'^(.+?)\s*[-|]\s*Forge$', re.I),
]

def clean_company_name(raw):
    if pd.isna(raw):
        return np.nan
    s = str(raw).strip()
    for pat in _NAME_PATTERNS:
        m = pat.match(s)
        if m:
            return m.group(1).strip()
    return s

# Verifica su casi noti
samples = [
    'Access 0x Stock and Financial Details - Forge',
    'Invest and Sell 100 Thieves Stock - Forge',
    'Invest and Sell [24]7.ai Stock - Forge',
    'Access OpenAI Stock and Financial Details - Forge',
    'Invest and Sell Anthropic Stock - Forge',
]
for s in samples:
    print(f'  {clean_company_name(s)}')

  0x
  100 Thieves
  [24]7.ai
  OpenAI
  Anthropic


In [5]:
# ── Categorizzazione round ────────────────────────────────────────────────────
# Raggruppa le varianti (Series A, Series A-1, Series A-2, ...) in categorie
# standard per le analisi. Manteniamo il nome originale nella colonna raw.

_ROUND_MAP = [
    (re.compile(r'pre.?seed', re.I),      'Pre-Seed'),
    (re.compile(r'seed',      re.I),      'Seed'),
    (re.compile(r'series\s+a', re.I),     'Series A'),
    (re.compile(r'series\s+b', re.I),     'Series B'),
    (re.compile(r'series\s+c', re.I),     'Series C'),
    (re.compile(r'series\s+d', re.I),     'Series D'),
    (re.compile(r'series\s+e', re.I),     'Series E'),
    (re.compile(r'series\s+f', re.I),     'Series F'),
    (re.compile(r'series\s+g', re.I),     'Series G'),
    (re.compile(r'series\s+[h-z]', re.I), 'Series H+'),
    (re.compile(r'series', re.I),         'Series Other'),
    (re.compile(r'ipo|public', re.I),     'IPO'),
    (re.compile(r'bridge|convert|note', re.I), 'Bridge/Conv.'),
    (re.compile(r'debt|loan|credit', re.I),    'Debt'),
    (re.compile(r'grant', re.I),          'Grant'),
]

def categorize_round(name):
    if pd.isna(name) or str(name).strip() in _NULL_VALUES:
        return 'Unknown'
    s = str(name).strip()
    for pat, cat in _ROUND_MAP:
        if pat.search(s):
            return cat
    return 'Other'

# Ordine logico per visualizzazioni
ROUND_ORDER = [
    'Pre-Seed','Seed','Series A','Series B','Series C',
    'Series D','Series E','Series F','Series G','Series H+',
    'Series Other','Bridge/Conv.','Debt','Grant','IPO','Other','Unknown'
]

print('Test categorizzazione:')
test_names = ['Series Seed', 'Series Seed-1', 'Series A', 'Series A-3',
              'Series B-2', 'Series C', 'Series D-1', 'Series E-2',
              'Series F', 'Series G', 'Series H', None, 'NA']
for n in test_names:
    print(f'  {str(n):20} → {categorize_round(n)}')

Test categorizzazione:
  Series Seed          → Seed
  Series Seed-1        → Seed
  Series A             → Series A
  Series A-3           → Series A
  Series B-2           → Series B
  Series C             → Series C
  Series D-1           → Series D
  Series E-2           → Series E
  Series F             → Series F
  Series G             → Series G
  Series H             → Series H+
  None                 → Unknown
  NA                   → Unknown


## 2 · Cleaning Companies

In [6]:
c = comp.copy()

# ── Nome azienda ──────────────────────────────────────────────────────────────
c['company_name'] = c['Company'].apply(clean_company_name)

# Verifica manuale
print('Campione nomi puliti:')
print(c['company_name'].head(10).tolist())

Campione nomi puliti:
['0x', '100 Thieves', '1047 Games', '10x Genomics, Inc.', '11x', '128 Technology', '1366 Technologies', '15Five', '1Kosmos', '1Life Healthcare']


In [7]:
# ── Anno di fondazione ────────────────────────────────────────────────────────
# 1900 è un placeholder di scraping per "anno non disponibile"
c['founded_year'] = c['Founded'].where(
    (c['Founded'] > 1900) & (c['Founded'] <= 2026),
    other=np.nan
).astype('Int64')   # Int64 nullable (supporta NaN con interi)

print(f'Founded validi : {c["founded_year"].notna().sum():,}')
print(f'Founded NaN    : {c["founded_year"].isna().sum()} ')
print(f'Range           : {c["founded_year"].min()} – {c["founded_year"].max()}')

Founded validi : 5,363
Founded NaN    : 7 
Range           : 1907 – 2026


In [8]:
# ── Sector / Subsector ────────────────────────────────────────────────────────
c['sector']    = c['Sector'].replace('Missing', np.nan)
c['subsector'] = c['Subsector'].replace('Missing', np.nan)

print('Sector distribution:')
print(c['sector'].value_counts(dropna=False).to_string())

Sector distribution:
sector
Enterprise Software     1847
Healthcare               986
Consumer & Lifestyle     759
Fintech                  657
Industrial               370
Transportation           182
Energy                   146
Foodtech                 115
Technology Hardware      113
Real Estate               98
Education                 91
NaN                        6


In [9]:
# ── Sede geografica ───────────────────────────────────────────────────────────
def parse_hq(hq):
    """Estrae city, state, country dalla stringa Headquarters."""
    if pd.isna(hq):
        return {'city': np.nan, 'state': np.nan, 'country': np.nan}
    parts = [p.strip() for p in str(hq).split(',')]
    country = parts[-1] if parts else np.nan
    city    = parts[0]  if len(parts) >= 1 else np.nan
    state   = None
    # Pattern specifico per USA: "City, ST, United States"
    m = re.search(r',\s*([A-Z]{2}),\s*United States', hq)
    if m:
        state = m.group(1)
    return {'city': city, 'state': state, 'country': country}

hq_parsed = c['Headquarters'].apply(parse_hq)
c['hq_country'] = [d['country'] for d in hq_parsed]
c['hq_state']   = [d['state']   for d in hq_parsed]   # solo USA
c['hq_city']    = [d['city']    for d in hq_parsed]

# Flag semplificato
c['is_usa'] = c['hq_country'].str.strip() == 'United States'

print('Top 10 paesi:')
print(c['hq_country'].value_counts().head(10).to_string())

Top 10 paesi:
hq_country
United States     4627
United Kingdom     143
Canada              70
Israel              66
Germany             58
China               45
France              43
India               32
Singapore           27
Sweden              21


In [10]:
# ── Investitori: lista strutturata ────────────────────────────────────────────
# Gli investitori in companies sono una stringa CSV → list
c['investor_list'] = c['Investors'].apply(
    lambda x: [i.strip() for i in str(x).split(',')
               if i.strip() and i.strip() not in _NULL_VALUES]
    if pd.notna(x) else []
)
c['n_investors'] = c['investor_list'].apply(len)

print('Distribuzione n_investors:')
print(c['n_investors'].describe())
print(f'Aziende senza investitori noti: {(c["n_investors"] == 0).sum():,}')

Distribuzione n_investors:
count    5370.000000
mean        4.213780
std         5.644437
min         0.000000
25%         0.000000
50%         1.000000
75%         7.000000
max        45.000000
Name: n_investors, dtype: float64
Aziende senza investitori noti: 2,533


In [11]:
# ── Selezione e rinomina colonne finali ──────────────────────────────────────
companies_clean = c[[
    'URL',
    'company_name',
    'sector',
    'subsector',
    'founded_year',
    'hq_country',
    'hq_state',
    'hq_city',
    'is_usa',
    'investor_list',
    'n_investors',
]].copy()

print(f'Companies clean: {companies_clean.shape}')
print()
companies_clean.head(3).T

Companies clean: (5370, 11)



,0,1,2
URL,https://forgeglobal.com/0x_stock/,https://forgeglobal.com/100-thieves_stock/,https://forgeglobal.com/1047-games_stock/
company_name,0x,100 Thieves,1047 Games
sector,Fintech,Consumer & Lifestyle,Consumer & Lifestyle
subsector,Other Fintech,Gaming,Gaming
founded_year,2016,2017,2016
hq_country,United States,United States,United States
hq_state,CA,CA,NV
hq_city,San Francisco,Los Angeles,Zephyr Cove
is_usa,True,True,True
investor_list,"[8 Decimal Capital, Zk Capital, Innovating Capital, James Sowers, Limitless ...",[],[]


## 3 · Cleaning Rounds

In [12]:
r = rounds.copy()

# ── Flag: round reale vs placeholder ─────────────────────────────────────────
# Le righe con Funding Date = NaN sono placeholder per aziende
# senza round dettagliati. Le manteniamo per non perdere il collegamento
# a companies, ma le flagghiamo.
r['has_round_data'] = r['Funding Date'].notna()

print(f'Round con dati reali    : {r["has_round_data"].sum():,}')
print(f'Round placeholder (vuoti): {(~r["has_round_data"]).sum():,}')

Round con dati reali    : 19,256
Round placeholder (vuoti): 2,282


In [13]:
# ── Date ──────────────────────────────────────────────────────────────────────
r['funding_date'] = pd.to_datetime(r['Funding Date'], format='%m/%d/%Y', errors='coerce')
r['funding_year']  = r['funding_date'].dt.year.astype('Int64')
r['funding_month'] = r['funding_date'].dt.month.astype('Int64')

ok = r['funding_date'].notna().sum()
print(f'Date parsate: {ok:,} / {r["has_round_data"].sum():,}')

Date parsate: 19,256 / 19,256


In [14]:
# ── Valori monetari ───────────────────────────────────────────────────────────
r['amount_raised_usd']  = r['Amount Raised'].apply(parse_money)
r['post_money_val_usd'] = r['Post-Money Valuation'].apply(parse_money)
r['price_per_share_usd'] = r['Price Per Share (Overview)'].apply(parse_money)

# Statistiche
real = r[r['has_round_data']]
print('Amount Raised (USD):')
print(real['amount_raised_usd'].describe().apply(lambda x: f'${x:,.0f}'))
print()
print('Post-Money Valuation (USD):')
print(real['post_money_val_usd'].describe().apply(lambda x: f'${x:,.0f}'))

Amount Raised (USD):
count             $19,236
mean          $64,988,333
std        $1,032,274,855
min                    $0
25%            $3,470,000
50%           $15,000,000
75%           $46,560,000
max      $122,000,000,000
Name: amount_raised_usd, dtype: str

Post-Money Valuation (USD):
count             $19,255
mean       $1,338,903,012
std       $20,348,476,110
min                $1,500
25%           $48,570,000
50%          $177,150,000
75%          $498,280,000
max      $965,000,000,000
Name: post_money_val_usd, dtype: str


In [16]:
# ── Round name categorization ─────────────────────────────────────────────────
r['round_name_raw']  = r['Round Name']
r['round_category']  = r['Round Name'].apply(categorize_round)

# Ordine categoriale
r['round_category'] = pd.Categorical(
    r['round_category'],
    categories=ROUND_ORDER,
    ordered=True
)

print('Round categories (round reali):')
counts = r.loc[r['has_round_data'], 'round_category'].value_counts().reindex(ROUND_ORDER, fill_value=0)
print(counts.to_string())

Round categories (round reali):
round_category
Pre-Seed          13
Seed            2279
Series A        6017
Series B        4019
Series C        2897
Series D        1671
Series E         874
Series F         463
Series G         189
Series H+        334
Series Other     403
Bridge/Conv.       5
Debt               0
Grant              0
IPO                0
Other             92
Unknown            0


In [17]:
# ── Termini preferenziali (liquidation, partecipazione, dividendi) ────────────
# Pulizia Liquidation Pref
def parse_multiplier(s):
    if pd.isna(s) or str(s).strip() in _NULL_VALUES | {'--'}:
        return np.nan
    m = re.match(r'([\d.]+)x', str(s).strip(), re.I)
    return float(m.group(1)) if m else np.nan

r['liq_pref_multiplier'] = r['Liquidation Pref As Multiplier'].apply(parse_multiplier)
r['is_participating']    = r['Participating'].str.strip().eq('Participating')
r['is_cumulative_div']   = r['Cumulative'].str.strip().eq('Cumulative')

# Dividend rate
def parse_pct(s):
    if pd.isna(s) or str(s).strip() in _NULL_VALUES | {'--'}:
        return np.nan
    m = re.match(r'([\d.]+)%', str(s).strip())
    return float(m.group(1)) / 100 if m else np.nan

r['dividend_rate'] = r['Dividend Rate'].apply(parse_pct)

# Conversion ratio
def parse_ratio(s):
    if pd.isna(s) or str(s).strip() in _NULL_VALUES | {'--'}:
        return np.nan
    m = re.match(r'([\d.]+)x', str(s).strip(), re.I)
    return float(m.group(1)) if m else np.nan

r['conversion_ratio'] = r['Conversion Ratio'].apply(parse_ratio)

print('Liq pref multiplier distribution:')
print(r['liq_pref_multiplier'].value_counts().head(8).to_string())
print(f'Participating rounds  : {r["is_participating"].sum():,}')
print(f'Cumulative dividend   : {r["is_cumulative_div"].sum():,}')

Liq pref multiplier distribution:
liq_pref_multiplier
1.0    18444
2.0      209
1.5      192
1.3       48
1.2       34
1.1       34
1.4       31
3.0       29
Participating rounds  : 906
Cumulative dividend   : 790


In [18]:
# ── Shares Outstanding (numerico) ─────────────────────────────────────────────
r['shares_outstanding'] = (
    r['Shares Outstanding']
    .str.replace(',', '', regex=False)
    .apply(lambda x: float(x) if pd.notna(x) and str(x).strip() not in _NULL_VALUES else np.nan)
)
# Valore '1' è anomalo (errore scraping) → NaN
r.loc[r['shares_outstanding'] == 1, 'shares_outstanding'] = np.nan

print('Shares outstanding (sample non-null):')
print(r['shares_outstanding'].dropna().describe())

Shares outstanding (sample non-null):
count    1.918500e+04
mean     1.791622e+07
std      3.783284e+08
min      3.000000e+00
25%      1.597194e+06
50%      4.684520e+06
75%      1.239325e+07
max      5.169134e+10
Name: shares_outstanding, dtype: float64


In [ ]:
# ── Investitori per round ─────────────────────────────────────────────────────
# Usiamo Key Investors; se assente, fallback su Key Investors (Overview)
r['investors_raw'] = r['Key Investors'].fillna(r['Key Investors (Overview)'])
r['investor_list_round'] = r['investors_raw'].apply(
    lambda x: [i.strip() for i in str(x).split(',')
               if i.strip() and i.strip() not in _NULL_VALUES | {'Undisclosed Investors'}]
    if pd.notna(x) else []
)
r['n_investors_round'] = r['investor_list_round'].apply(len)

print('n_investors_round (round reali):')
print(r[r['has_round_data']]['n_investors_round'].describe())


n_investors_round (round reali):


KeyError: 'n_investors_round'

In [ ]:
# ── Selezione colonne finali rounds ───────────────────────────────────────────
rounds_clean = r[[
    'URL',
    'has_round_data',
    'funding_date',
    'funding_year',
    'funding_month',
    'round_name_raw',
    'round_category',
    'amount_raised_usd',
    'post_money_val_usd',
    'price_per_share_usd',
    'shares_outstanding',
    'liq_pref_multiplier',
    'is_participating',
    'is_cumulative_div',
    'dividend_rate',
    'conversion_ratio',
    'investor_list_round',
    'n_investors_round',
]].copy()

print(f'Rounds clean: {rounds_clean.shape}')
rounds_clean.head(3).T

## 4 · Ricalcolo Total Funding in Companies

In [ ]:
# Ricaviamo il totale dai round perché il campo nativo era vuoto al 100%.
# Sommiamo gli Amount Raised per URL, ignorando i round senza importo.

total_funding = (
    rounds_clean[rounds_clean['has_round_data']]
    .groupby('URL')['amount_raised_usd']
    .sum(min_count=1)          # NaN se tutti i round hanno NaN
    .rename('total_funding_usd')
)

companies_clean = companies_clean.merge(total_funding, on='URL', how='left')

print(f'Companies con Total Funding derivato: '
      f'{companies_clean["total_funding_usd"].notna().sum():,} '
      f'/ {len(companies_clean):,}')
print()
print('Distribuzione Total Funding (USD):')
print(companies_clean['total_funding_usd'].describe().apply(lambda x: f'${x:,.0f}'))

## 5 · Controlli qualità post-cleaning

In [ ]:
# ── Companies ─────────────────────────────────────────────────────────────────
print('=== COMPANIES — copertura post-cleaning ===')
for col in companies_clean.columns:
    if col == 'investor_list':
        n_non_empty = (companies_clean[col].apply(len) > 0).sum()
        print(f'  {col:25} : {n_non_empty:,} non-empty ({n_non_empty/len(companies_clean)*100:.1f}%)')
    else:
        n_valid = companies_clean[col].notna().sum()
        if companies_clean[col].dtype == bool:
            n_valid = companies_clean[col].sum()
        print(f'  {col:25} : {n_valid:,} ({n_valid/len(companies_clean)*100:.1f}%)')

In [ ]:
# ── Rounds reali ──────────────────────────────────────────────────────────────
real_clean = rounds_clean[rounds_clean['has_round_data']]
print(f'=== ROUNDS REALI — {len(real_clean):,} righe ===')
for col in real_clean.columns:
    if 'list' in col:
        continue
    n = real_clean[col].notna().sum() if real_clean[col].dtype != bool else real_clean[col].sum()
    print(f'  {col:30} : {n:,} ({n/len(real_clean)*100:.1f}%)')

## 6 · Salvataggio

In [ ]:
# Parquet è più efficiente di CSV per colonne con liste e tipi misti.
# Se non hai pyarrow installato, usa .to_csv() con index=False.

try:
    companies_clean.to_parquet(OUT_DIR / 'companies_clean.parquet', index=False)
    rounds_clean.to_parquet(OUT_DIR / 'rounds_clean.parquet', index=False)
    print('Salvato in formato Parquet:')
    print(f'  companies_clean.parquet ({companies_clean.shape})')
    print(f'  rounds_clean.parquet    ({rounds_clean.shape})')
except ImportError:
    # fallback CSV
    companies_clean_csv = companies_clean.drop(columns=['investor_list'])
    rounds_clean_csv    = rounds_clean.drop(columns=['investor_list_round'])
    companies_clean_csv.to_csv(OUT_DIR / 'companies_clean.csv', index=False)
    rounds_clean_csv.to_csv(OUT_DIR / 'rounds_clean.csv', index=False)
    print('Salvato in formato CSV (senza colonne lista):')
    print(f'  companies_clean.csv ({companies_clean_csv.shape})')
    print(f'  rounds_clean.csv    ({rounds_clean_csv.shape})')

In [ ]:
print('\n=== CLEANING COMPLETATO ===')
print(f'Companies clean : {companies_clean.shape[0]:,} righe, {companies_clean.shape[1]} colonne')
print(f'Rounds clean    : {rounds_clean.shape[0]:,} righe, {rounds_clean.shape[1]} colonne')
print(f'  di cui reali  : {real_clean.shape[0]:,}')
print(f'  placeholder   : {(~rounds_clean["has_round_data"]).sum():,}')